# Calibration Loss for ANCOVA NPE Training

This notebook compares two training approaches for the ANCOVA 2-arm continuous outcome model:

1. **Baseline** — Standard BayesFlow training with negative log-likelihood (NLL) loss only
2. **Calibrated** — NLL + differentiable calibration loss from [Falkner et al. (NeurIPS 2023)](https://arxiv.org/abs/2310.13402)

Both models use the **same architecture and training settings**. The only difference is the
calibration loss term, which penalizes under-coverage during training.

The calibration loss package lives at https://github.com/matthiaskloft/bayesflow_calibration_loss and is installed via:
```bash
pip install -e ".[calibration]"
```

In [1]:
%pip install -e ".."
%pip install -e "../../bayesflow_calibration_loss"

import os
import sys
import inspect

if not os.environ.get("KERAS_BACKEND"):
    os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import matplotlib.pyplot as plt
from itertools import product

import keras
import bayesflow as bf
import bayesflow_calibration_loss as bfc

# ANCOVA model
from bayesflow_rct.models.ancova.model import (
    ANCOVAConfig,
    create_ancova_adapter,
    create_simulator,
    create_ancova_workflow_components,
    create_validation_grid,
    make_simulate_fn,
)
from bayesflow_rct.core.infrastructure import (
    build_summary_network,
    build_inference_network,
)
from bayesflow_rct.core.utils import MovingAverageEarlyStopping
from bayesflow_rct.core.validation import (
    run_validation_pipeline,
    make_bayesflow_infer_fn,
)

# Calibration loss (from bayesflow_calibration_loss)
from bayesflow_calibration_loss import (
    CalibratedContinuousApproximator,
    CalibrationMonitorCallback,
)

required_calibration_kwargs = (
    "target_calibration_error",
    "lr_lambda",
    "lambda_max",
    "normalization_burn_in",
    "batch_size_calibration",
)
init_params = inspect.signature(CalibratedContinuousApproximator.__init__).parameters
missing_kwargs = [name for name in required_calibration_kwargs if name not in init_params]
if missing_kwargs:
    raise RuntimeError(
        "Loaded bayesflow_calibration does not match expected Lagrangian API. "
        f"Missing __init__ kwargs: {missing_kwargs}. "
        "Run this cell again, then restart kernel and re-run from Cell 2."
    )

np.set_printoptions(suppress=True)
RNG = np.random.default_rng(2025)

print(f"Python executable: {sys.executable}")
print(f"bayesflow_calibration source: {bfc.__file__}")
print(f"BayesFlow {bf.__version__}")
print(f"Keras {keras.__version__} (backend: {keras.backend.backend()})")
print("Lagrangian calibration API check passed")

Obtaining file:///C:/Users/Matze/Documents/GitHub/bayesflow_rct
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for bayesflow_rct (pyproject.toml): started
  Building editable for bayesflow_rct (pyproject.toml): finished with status 'done'
  Created wheel for bayesflow_rct: filename=bayesflow_rct-0.1.0-0.editable-py3-none-any.whl size=2700 sha256=2c0e08a03871d7c31aba10a1c3a07eed2f0a48cbdc65d59f1a8684cea9a323e8
  Stored in directory: C:\Users\Matze\AppData\Local\Temp\pip-ephem-wheel-cache-i4bpmwqg\wheels\db\89\5e\8ee7869f6119d2


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Obtaining file:///C:/Users/Matze/Documents/GitHub/bayesflow_calibration_loss
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for bayesflow-calibration-loss (pyproject.toml): started
  Building editable for bayesflow-calibration-loss (pyproject.toml): finished with status 'done'
  Created wheel for bayesflow-calibration-loss: filename=bayesflow_calibration_loss-0.1.0-0.editable-py3-none-any.whl size=4880 sha256=2456d00a8c3a2184267f4ca44f49a6cb46a0a287820d5b44fd4a7f76026b5911
  Stored in directory: C:\Users\Matze\AppData\Local\T


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
INFO:bayesflow:Using backend 'torch'
When using torch backend, we need to disable autograd by default to avoid excessive memory usage. Use

with torch.enable_grad():
    ...

in contexts where you need gradients (e.g. custom training loops).


Python executable: c:\Users\Matze\Documents\GitHub\bayesflow_rct\venv-py312\Scripts\python.exe
bayesflow_calibration source: C:\Users\Matze\Documents\GitHub\bayesflow_calibration_loss\src\bayesflow_calibration_loss\__init__.py
BayesFlow 2.0.7
Keras 3.12.0 (backend: torch)
Lagrangian calibration API check passed


## 1. Shared Configuration

Both models share the same ANCOVA config, simulator, adapter, and network architecture.
For this notebook we use a smaller NPE architecture to improve stability and runtime:

- Summary network: `summary_dim=6`, `depth=2`, `width=32`, `dropout=0.05`
- Inference network: `depth=4`, `hidden_sizes=(64, 64)`, `dropout=0.10`

In [2]:
config = ANCOVAConfig()

# Smaller, more stable architecture for this notebook
config.workflow.summary_network.summary_dim = 6
config.workflow.summary_network.depth = 2
config.workflow.summary_network.width = 32
config.workflow.summary_network.dropout = 0.05

config.workflow.inference_network.network_type = "FlowMatching"
config.workflow.inference_network.widths = (128, 128, 128)
config.workflow.inference_network.dropout = 0.10

# Keep training defaults unless explicitly changed elsewhere
print("Summary network:", config.workflow.summary_network)
print("Inference network:", config.workflow.inference_network)
print("Training:", config.workflow.training)

Summary network: SummaryNetworkConfig(summary_dim=6, depth=2, width=32, dropout=0.05, network_type='DeepSet')
Inference network: InferenceNetworkConfig(network_type='FlowMatching', depth=7, hidden_sizes=(128, 128), widths=(128, 128, 128), use_optimal_transport=False, dropout=0.1)
Training: TrainingConfig(initial_lr=0.0007, decay_rate=0.85, batch_size=320, epochs=200, batches_per_epoch=50, validation_sims=1000, early_stopping_patience=10, early_stopping_window=10)


In [3]:
# Create simulator and adapter (shared between both models)
simulator = create_simulator(config, RNG)
adapter = create_ancova_adapter()

# Quick test
sim_test = simulator.sample(10)
processed = adapter(sim_test)
print("summary_variables:", processed["summary_variables"].shape)
print("inference_variables:", processed["inference_variables"].shape)
print("inference_conditions:", processed["inference_conditions"].shape)

summary_variables: (10, 979, 3)
inference_variables: (10, 1)
inference_conditions: (10, 4)


## 2. Shared Training Settings

In [4]:
# Training hyperparameters (same for both models)
EPOCHS = 50
BATCH_SIZE = 320
BATCHES_PER_EPOCH = 100
VALIDATION_SIMS = 500

train_config = config.workflow.training


def make_optimizer():
    """Create a fresh optimizer (must be separate per model)."""
    steps_per_epoch = BATCH_SIZE * BATCHES_PER_EPOCH
    lr_schedule = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=train_config.initial_lr,
        decay_steps=steps_per_epoch,
        decay_rate=train_config.decay_rate,
        staircase=True,
    )
    return keras.optimizers.Adam(learning_rate=lr_schedule)


def make_early_stopping():
    """Create a fresh early stopping callback."""
    return MovingAverageEarlyStopping(
        window=train_config.early_stopping_window,
        patience=train_config.early_stopping_patience,
    )

## 3. Train Baseline Model (NLL only)

Standard BayesFlow training — the normalizing flow is trained to maximize the log-density
of the true parameters under the learned posterior.

In [5]:
# Build fresh networks for the baseline
summary_net_base, inference_net_base, _ = create_ancova_workflow_components(config)

workflow_base = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    inference_network=inference_net_base,
    summary_network=summary_net_base,
    optimizer=make_optimizer(),
    inference_conditions=["N", "p_alloc", "prior_df", "prior_scale"],
)
workflow_base.approximator.compile(optimizer=make_optimizer())

print("Baseline workflow created")

Baseline workflow created


In [6]:
history_base = workflow_base.fit_online(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    num_batches_per_epoch=BATCHES_PER_EPOCH,
    validation_data=VALIDATION_SIMS
)
print("Baseline training complete")

INFO:bayesflow:Fitting on dataset instance of OnlineDataset.
INFO:bayesflow:Building on a test batch.


Epoch 1/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 20s 197ms/step - loss: 31.2317 - val_loss: 0.9361
Epoch 2/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 20s 196ms/step - loss: 7.4410 - val_loss: 2.2552
Epoch 3/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 179ms/step - loss: 3.7826 - val_loss: 1.9542
Epoch 4/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 181ms/step - loss: 2.5117 - val_loss: 1.1388
Epoch 5/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 179ms/step - loss: 2.2098 - val_loss: 0.6334
Epoch 6/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 180ms/step - loss: 1.7318 - val_loss: 0.7283
Epoch 7/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 19s 185ms/step - loss: 1.8259 - val_loss: 0.7068
Epoch 8/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 184ms/step - loss: 1.8751 - val_loss: 0.5714
Epoch 9/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 181ms/step - loss: 4.7786 - val_loss: 0.5048
Epoch 10/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 184ms/step - loss: 1.7428 - val_loss: 0.5916
Epoch 11/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 19s 187ms/step - loss: 1.5495 - val_loss: 0.2793
Epoch 12/50
100/10

## 4. Train Calibrated Model (NLL + Calibration Loss)

Uses `CalibratedContinuousApproximator` from the `bayesflow-calibration` package.
This subclasses BayesFlow's `ContinuousApproximator` and injects a calibration loss
term that penalizes under-coverage during training.

With the new API, calibration pressure is controlled by **Lagrangian dual ascent**:
**`total_loss = nll_loss + gamma * calibration_loss`**

where `gamma` is adapted online via `lambda` to satisfy a target calibration error (`epsilon`).

Key settings:
- **`start_epoch`**: warmup with pure NLL before activating calibration
- **`target_calibration_error`**: desired calibration-error level (constraint target)
- **`lr_lambda`**: adaptation speed for `lambda`
- **`lambda_max`**: cap on calibration pressure to protect NLL fit
- **`normalize_loss=True`** + **`normalization_burn_in`**: scale stabilization
- **`calibration_mode=0.0`**: conservativeness mode (penalize under-coverage only)
- **`n_samples=100`**: posterior samples for rank computation
- **`batch_size_calibration=64`**: calibration sub-batch size (reduces overhead)

In [7]:
# Build fresh networks for the calibrated model
summary_net_cal, inference_net_cal, _ = create_ancova_workflow_components(config)

# Sanity-check condition decoding used by simulator-auto-wrapped calibration prior
sim_check = simulator.sample(2000)
adapted_check = adapter(sim_check)
cond_check = np.asarray(adapted_check["inference_conditions"] )

true_df = np.asarray(sim_check["prior_df"]).reshape(-1).astype(int)
true_scale = np.asarray(sim_check["prior_scale"]).reshape(-1).astype(float)
dec_df = np.clip(np.rint(np.expm1(cond_check[:, 2])).astype(int), 0, config.meta.prior_df_max)
dec_scale = cond_check[:, 3].astype(float)

df_match = float(np.mean(dec_df == true_df))
scale_mae = float(np.mean(np.abs(dec_scale - true_scale)))

print("Condition decode check (for simulator auto-wrap):")
print(f"  prior_df exact match rate: {df_match:.3f}")
print(f"  prior_scale MAE: {scale_mae:.6f}")
if df_match < 0.99 or scale_mae > 1e-4:
    print("  WARNING: decoding mismatch detected.")
else:
    print("  Decoding looks consistent.")

Condition decode check (for simulator auto-wrap):
  prior_df exact match rate: 1.000
  prior_scale MAE: 0.000000
  Decoding looks consistent.


In [8]:
# Create the calibrated approximator
CAL_START_EPOCH = max(3, int(round(0.3 * EPOCHS)))
CAL_MODE = 1.0 # penalize under- and over-confidence equally
TARGET_CAL_ERROR = 0.01
LR_LAMBDA = 0.01
LAMBDA_MAX = 1.0
NORMALIZATION_BURN_IN = 1000

print(
    f"Planned calibration activation: start_epoch={CAL_START_EPOCH} "
    f"(out of {EPOCHS} epochs)"
 )

if not (0 <= CAL_START_EPOCH <= EPOCHS):
    raise ValueError(
        f"Invalid start epoch for EPOCHS={EPOCHS}: start={CAL_START_EPOCH}"
    )

approximator_cal = CalibratedContinuousApproximator(
    # BayesFlow ContinuousApproximator args
    inference_network=inference_net_cal,
    summary_network=summary_net_cal,
    adapter=adapter,
    # Calibration-specific args (new Lagrangian API)
    start_epoch=CAL_START_EPOCH,
    target_calibration_error=TARGET_CAL_ERROR,
    lr_lambda=LR_LAMBDA,
    lambda_max=LAMBDA_MAX,
    normalize_loss=True,
    normalization_burn_in=NORMALIZATION_BURN_IN,
    calibration_mode=CAL_MODE,
    n_samples=100,
    batch_size_calibration=64,
 )

approximator_cal.compile(optimizer=make_optimizer())
print("Calibrated approximator created")
print(f"  start_epoch: {approximator_cal.start_epoch}")
print(f"  target_calibration_error: {approximator_cal.target_calibration_error}")
print(f"  lr_lambda: {approximator_cal.lr_lambda}")
print(f"  lambda_max: {approximator_cal.lambda_max}")
print(f"  normalize_loss: {approximator_cal.normalize_loss}")
print(f"  normalization_burn_in: {approximator_cal.normalization_burn_in}")
print(f"  calibration_mode: {approximator_cal.calibration_mode}")
print(f"  n_samples: {approximator_cal.n_samples}")
print(f"  batch_size_calibration: {approximator_cal.batch_size_calibration}")

Planned calibration activation: start_epoch=15 (out of 50 epochs)
Calibrated approximator created
  start_epoch: 15
  target_calibration_error: 0.01
  lr_lambda: 0.01
  lambda_max: 1.0
  normalize_loss: True
  normalization_burn_in: 1000
  calibration_mode: 1.0
  n_samples: 100
  batch_size_calibration: 64


In [9]:
# We can't use BasicWorkflow for the calibrated model since it uses
# ContinuousApproximator internally. Instead, we train the approximator
# directly using BayesFlow's online training loop.

# Create a BasicWorkflow just for the data pipeline, but swap the approximator
workflow_cal = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    inference_network=inference_net_cal,
    summary_network=summary_net_cal,
    optimizer=make_optimizer(),
    inference_conditions=["N", "p_alloc", "prior_df", "prior_scale"],
)
# Replace the approximator with our calibrated one
workflow_cal.approximator = approximator_cal

# Train with CalibrationMonitorCallback (REQUIRED for start_epoch gating + lambda updates)
history_cal = workflow_cal.fit_online(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    num_batches_per_epoch=BATCHES_PER_EPOCH,
    validation_data=VALIDATION_SIMS,
    callbacks=[CalibrationMonitorCallback()],
)
print("Calibrated training complete")

INFO:bayesflow:Fitting on dataset instance of OnlineDataset.
INFO:bayesflow:Building on a test batch.
INFO:bayesflow_calibration_loss.diagnostics:Calibration loss normalization enabled (burn_in=1000)


Epoch 1/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step - gamma: 0.0000e+00 - lambda: 0.0000e+00 - loss: 5.6382

INFO:bayesflow_calibration_loss.diagnostics:Epoch 0: lambda=0.0000, target_epsilon=0.010000


100/100 ━━━━━━━━━━━━━━━━━━━━ 59s 589ms/step - gamma: 0.0000e+00 - lambda: 0.0000e+00 - loss: 5.6382 - val_calibration_error: 0.0585 - val_calibration_loss: 0.0585 - val_loss: 3.2119
Epoch 2/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step - gamma: 0.0000e+00 - lambda: 0.0000e+00 - loss: 1.7880

INFO:bayesflow_calibration_loss.diagnostics:Epoch 1: lambda=0.0000, target_epsilon=0.010000


...

INFO:bayesflow_calibration_loss.diagnostics:Epoch 23: lambda=0.0000, target_epsilon=0.010000


100/100 ━━━━━━━━━━━━━━━━━━━━ 65s 650ms/step - gamma: 0.0000e+00 - lambda: 0.0000e+00 - loss: 0.3981 - val_calibration_error: 0.1818 - val_calibration_loss: 0.1818 - val_loss: 1.1415
Epoch 25/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 442ms/step - gamma: 0.0000e+00 - lambda: 0.0000e+00 - loss: 0.3008

INFO:bayesflow_calibration_loss.diagnostics:Epoch 24: lambda=0.0000, target_epsilon=0.010000


100/100 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - gamma: 0.0000e+00 - lambda: 0.0000e+00 - loss: 0.1953 - val_calibration_error: 0.0568 - val_calibration_loss: 0.0568 - val_loss: 1.1079
Epoch 26/50
  1/100 ━━━━━━━━━━━━━━━━━━━━ 1:37:20 59s/step - calibration_error: 0.3344 - calibration_loss: 0.3344 - effective_gamma: 0.0063 - gamma: 0.0063 - lambda: 0.0032 - loss: 0.2529

OutOfMemoryError: Exception encountered when calling Dense.call().

[1mCUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 19.48 GiB is allocated by PyTorch, and 1.13 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)[0m

Arguments received by Dense.call():
  • inputs=torch.Tensor(shape=torch.Size([6400, 128]), dtype=float32)
  • training=False

## 5. Training Loss Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Training loss
ax = axes[0]
ax.plot(history_base.history["loss"], label="Baseline (NLL)", alpha=0.8)
ax.plot(history_cal.history["loss"], label="Calibrated (NLL + cal)", alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss")
ax.set_title("Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Validation loss
ax = axes[1]
ax.plot(history_base.history.get("val_loss", []), label="Baseline", alpha=0.8)
ax.plot(history_cal.history.get("val_loss", []), label="Calibrated", alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Loss")
ax.set_title("Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Calibration error history (calibrated model only)
ax = axes[2]
cal_train = history_cal.history.get("calibration_error", history_cal.history.get("calibration_loss", []))
cal_val = history_cal.history.get("val_calibration_error", history_cal.history.get("val_calibration_loss", []))

if len(cal_train) > 0:
    ax.plot(cal_train, label="Train calibration error", alpha=0.9)
if len(cal_val) > 0:
    ax.plot(cal_val, label="Validation calibration error", alpha=0.9)

ax.set_xlabel("Epoch")
ax.set_ylabel("Calibration Error")
ax.set_title("Calibration Error History")
if len(cal_train) > 0 or len(cal_val) > 0:
    ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. BayesFlow Built-in Diagnostics

Quick comparison using BayesFlow's default diagnostic metrics.

In [ ]:
# Generate shared validation data
val_sims = simulator.sample(1000)

# Compute default diagnostics for both models
metrics_base = workflow_base.compute_default_diagnostics(test_data=val_sims)
metrics_cal = workflow_cal.compute_default_diagnostics(test_data=val_sims)

import pandas as pd

comparison = pd.DataFrame({
    "Baseline": metrics_base["b_group"],
    "Calibrated": metrics_cal["b_group"],
})
print("BayesFlow default diagnostics (b_group):")
display(comparison)

## 7. Validation on Conditions Grid

Systematic comparison across a grid of ANCOVA conditions (N, prior_df, prior_scale, b_group).

In [ ]:
# Use the reduced validation grid (faster)
conditions = create_validation_grid(extended=False)
print(f"Validation grid: {len(conditions)} conditions")
print(f"Example: {conditions[0]}")

In [ ]:
# Run validation for baseline model
simulate_fn = make_simulate_fn(rng=RNG)

infer_fn_base = make_bayesflow_infer_fn(
    workflow_base.approximator,
    param_key="b_group",
    data_keys=["outcome", "covariate", "group"],
    context_keys={"N": int, "p_alloc": float, "prior_df": float, "prior_scale": float},
)

print("=== Baseline Model ===")
results_base = run_validation_pipeline(
    conditions_list=conditions,
    n_sims=500,
    n_post_draws=500,
    simulate_fn=simulate_fn,
    infer_fn=infer_fn_base,
    true_param_key="b_arm_treat",
    verbose=True,
)

In [ ]:
# Run validation for calibrated model
infer_fn_cal = make_bayesflow_infer_fn(
    workflow_cal.approximator,
    param_key="b_group",
    data_keys=["outcome", "covariate", "group"],
    context_keys={"N": int, "p_alloc": float, "prior_df": float, "prior_scale": float},
)

print("=== Calibrated Model ===")
results_cal = run_validation_pipeline(
    conditions_list=conditions,
    n_sims=500,
    n_post_draws=500,
    simulate_fn=simulate_fn,
    infer_fn=infer_fn_cal,
    true_param_key="b_arm_treat",
    verbose=True,
)

## 8. Summary Comparison

In [ ]:
s_base = results_base["metrics"]["summary"]
s_cal = results_cal["metrics"]["summary"]

summary_keys = [
    "recovery_corr", "recovery_r2", "overall_nrmse", "overall_bias",
    "mean_contraction", "mean_cal_error",
    "coverage_50", "coverage_80", "coverage_90", "coverage_95",
    "sbc_ks_pvalue", "sbc_c2st_accuracy",
]

comparison_df = pd.DataFrame({
    "Metric": summary_keys,
    "Baseline": [s_base.get(k, float("nan")) for k in summary_keys],
    "Calibrated": [s_cal.get(k, float("nan")) for k in summary_keys],
}).set_index("Metric")

# Format nicely
comparison_df["Difference"] = comparison_df["Calibrated"] - comparison_df["Baseline"]

print("=" * 65)
print("         Baseline  vs  Calibrated  (NLL + calibration loss)")
print("=" * 65)
display(comparison_df.round(4))

print("\nKey:")
print("  mean_cal_error: lower is better (0 = perfect calibration)")
print("  coverage_XX: closer to XX/100 is better")
print("  sbc_ks_pvalue: > 0.05 suggests calibration")
print("  sbc_c2st_accuracy: closer to 0.5 is better")

## 9. Coverage Profile Comparison

The coverage profile shows empirical vs. nominal coverage at every level from 1% to 99%.
A well-calibrated model follows the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (label, results) in zip(axes, [("Baseline", results_base), ("Calibrated", results_cal)]):
    profile = results["metrics"]["summary"]["coverage_profile"]
    nominal = sorted(profile.keys())
    empirical = [profile[n] for n in nominal]

    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
    ax.plot(nominal, empirical, "b-", lw=2, label="Empirical")
    ax.fill_between(nominal, empirical, nominal, alpha=0.2, color="red")
    ax.set_xlabel("Nominal Coverage")
    ax.set_ylabel("Empirical Coverage")
    ax.set_title(f"{label} — Coverage Profile")
    ax.legend(loc="upper left")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

    cal_err = results["metrics"]["summary"]["mean_cal_error"]
    ax.text(0.95, 0.05, f"Cal. Error: {cal_err:.4f}",
            transform=ax.transAxes, ha="right", va="bottom",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

plt.tight_layout()
plt.show()

## 10. Per-Condition Calibration Error Comparison

In [ ]:
cond_base = results_base["metrics"]["condition_metrics"]
cond_cal = results_cal["metrics"]["condition_metrics"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mean calibration error per condition
ax = axes[0]
x = np.arange(len(cond_base))
w = 0.35
ax.bar(x - w/2, cond_base["mean_cal_error"], w, label="Baseline", alpha=0.8)
ax.bar(x + w/2, cond_cal["mean_cal_error"], w, label="Calibrated", alpha=0.8)
ax.set_xlabel("Condition")
ax.set_ylabel("Mean Calibration Error")
ax.set_title("Calibration Error by Condition")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# 95% coverage per condition
ax = axes[1]
ax.bar(x - w/2, cond_base["coverage_95"], w, label="Baseline", alpha=0.8)
ax.bar(x + w/2, cond_cal["coverage_95"], w, label="Calibrated", alpha=0.8)
ax.axhline(0.95, color="red", ls="--", alpha=0.7, label="Nominal (95%)")
ax.set_xlabel("Condition")
ax.set_ylabel("95% Coverage")
ax.set_title("95% Coverage by Condition")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# NRMSE per condition (should not degrade much)
ax = axes[2]
ax.bar(x - w/2, cond_base["nrmse"], w, label="Baseline", alpha=0.8)
ax.bar(x + w/2, cond_cal["nrmse"], w, label="Calibrated", alpha=0.8)
ax.set_xlabel("Condition")
ax.set_ylabel("NRMSE")
ax.set_title("NRMSE by Condition (lower is better)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 11. Condition-Level Summary Tables

In [ ]:
print("Baseline — condition summary:")
display(results_base["metrics"]["condition_summary"].round(4))

print("\nCalibrated — condition summary:")
display(results_cal["metrics"]["condition_summary"].round(4))

## 12. Conclusion

**Expected observations:**

- The calibrated model should have **lower calibration error** and empirical coverage
  closer to nominal levels (especially at 90% and 95%).
- In conservativeness mode (`calibration_mode=0.0`), the calibrated model may produce
  slightly **wider** credible intervals (higher posterior SD) — this is by design.
- The NRMSE and recovery correlation should remain similar, showing the calibration loss
  does not significantly harm point estimation accuracy.
- The SBC KS p-value should be higher for the calibrated model (closer to uniform rank
  distribution), and the C2ST accuracy closer to 0.5.

**Notes:**
- The calibration loss adds training overhead (roughly 2-6x depending on `n_samples`
  and `batch_size_calibration`). Adjust these for your compute budget.
- Use Lagrangian warmup (`start_epoch`) so training begins with pure NLL, then calibration
  pressure is increased automatically through `lambda` to match `target_calibration_error`.